# M6 — Specialist LoRA Merging (Zero Training Cost)

Merges LoRA adapters from 3 Phase 1 specialists:
- **Adapter A** = E3 (CW-WSFT) → best decision accuracy
- **Adapter B** = E5a (decision entropy SFT) → best safety
- **Adapter C** = E7 (decision-only SFT) → best reasoning

No GPU training needed — just linear combination of weight tensors.

**⚠️ Before running:** copy your Phase 1 checkpoint folders to `~/kd_project/data/`

In [1]:
# Cell 0: Config — SET YOUR PATHS
import os, sys, json
import torch
from typing import Dict

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import RUNS_DIR, STUDENTS

OUT_DIR = os.path.join(RUNS_DIR, "m6_lora_merging")
os.makedirs(OUT_DIR, exist_ok=True)

# ⚠️ UPDATE to where your Phase 1 checkpoints are
PHASE1_BASE = os.path.expanduser("data")

SPECIALIST_DIRS = {
    "accuracy":  os.path.join(PHASE1_BASE, "cw_wsft_runs", "{student}"),
    "safety":    os.path.join(PHASE1_BASE, "e5a_decision_entropy_sft", "{student}"),
    "reasoning": os.path.join(PHASE1_BASE, "e7_decision_only_sft", "{student}"),
}

# Default merge weights
MERGE_WEIGHTS = {"accuracy": 0.4, "safety": 0.3, "reasoning": 0.3}

# Extra combos to sweep (free — merging takes <1 sec)
WEIGHT_SWEEP = [
    {"accuracy": 0.4, "safety": 0.3, "reasoning": 0.3},
    {"accuracy": 0.5, "safety": 0.25, "reasoning": 0.25},
    {"accuracy": 0.33, "safety": 0.33, "reasoning": 0.34},
    {"accuracy": 0.3, "safety": 0.2, "reasoning": 0.5},
]

# Check which specialists exist
for student_name in STUDENTS:
    print(f"\n{student_name}:")
    for spec, template in SPECIALIST_DIRS.items():
        path = template.format(student=student_name)
        exists = os.path.exists(path)
        print(f"  {spec}: {'✅' if exists else '❌'} {path}")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



qwen25_1p5b_base:
  accuracy: ✅ data\cw_wsft_runs\qwen25_1p5b_base
  safety: ✅ data\e5a_decision_entropy_sft\qwen25_1p5b_base
  reasoning: ✅ data\e7_decision_only_sft\qwen25_1p5b_base

qwen25_3b_base:
  accuracy: ✅ data\cw_wsft_runs\qwen25_3b_base
  safety: ✅ data\e5a_decision_entropy_sft\qwen25_3b_base
  reasoning: ✅ data\e7_decision_only_sft\qwen25_3b_base

qwen25_7b_base:
  accuracy: ✅ data\cw_wsft_runs\qwen25_7b_base
  safety: ✅ data\e5a_decision_entropy_sft\qwen25_7b_base
  reasoning: ✅ data\e7_decision_only_sft\qwen25_7b_base


In [2]:
# Cell 1: Merging functions
import shutil
from transformers import AutoTokenizer

def load_lora_state_dict(adapter_path: str) -> Dict[str, torch.Tensor]:
    safetensors_path = os.path.join(adapter_path, "adapter_model.safetensors")
    bin_path = os.path.join(adapter_path, "adapter_model.bin")
    if os.path.exists(safetensors_path):
        from safetensors.torch import load_file
        return load_file(safetensors_path)
    elif os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No adapter in {adapter_path}")

def merge_lora_weights(specialist_paths, weights):
    state_dicts = {}
    for name, path in specialist_paths.items():
        state_dicts[name] = load_lora_state_dict(path)
        print(f"  Loaded {name}: {len(state_dicts[name])} tensors")

    keys = set(state_dicts[list(state_dicts.keys())[0]].keys())
    merged = {}
    for key in keys:
        tensors = [weights[name] * state_dicts[name][key].float()
                    for name in specialist_paths if key in state_dicts[name]]
        merged[key] = sum(tensors)
    print(f"  ✅ Merged {len(merged)} params with {weights}")
    return merged

print("✅ Merging functions ready")

✅ Merging functions ready


In [3]:
# Cell 2: Merge default weights for each student
from safetensors.torch import save_file

for student_name in STUDENTS:
    print(f"\n{'='*50} M6: {student_name} {'='*50}")

    specialist_paths = {spec: template.format(student=student_name)
                        for spec, template in SPECIALIST_DIRS.items()}

    if not all(os.path.exists(p) for p in specialist_paths.values()):
        print("  ⚠️ Missing checkpoints, skipping")
        continue

    merged = merge_lora_weights(specialist_paths, MERGE_WEIGHTS)

    out_path = os.path.join(OUT_DIR, student_name)
    os.makedirs(out_path, exist_ok=True)

    shutil.copy2(os.path.join(specialist_paths["accuracy"], "adapter_config.json"),
                 os.path.join(out_path, "adapter_config.json"))
    save_file(merged, os.path.join(out_path, "adapter_model.safetensors"))

    tokenizer = AutoTokenizer.from_pretrained(STUDENTS[student_name]["path"], trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    tokenizer.save_pretrained(out_path)

    print(f"  ✅ Saved → {out_path}")

print("\n✅ M6 default merging complete.")


================================================== M6: qwen25_1p5b_base ==================================================
  Loaded accuracy: 224 tensors
  Loaded safety: 224 tensors
  Loaded reasoning: 224 tensors
  ✅ Merged 224 params with {'accuracy': 0.4, 'safety': 0.3, 'reasoning': 0.3}


  ✅ Saved → runs\m6_lora_merging\qwen25_1p5b_base

================================================== M6: qwen25_3b_base ==================================================
  Loaded accuracy: 288 tensors
  Loaded safety: 288 tensors
  Loaded reasoning: 288 tensors
  ✅ Merged 288 params with {'accuracy': 0.4, 'safety': 0.3, 'reasoning': 0.3}
  ✅ Saved → runs\m6_lora_merging\qwen25_3b_base

================================================== M6: qwen25_7b_base ==================================================
  Loaded accuracy: 224 tensors
  Loaded safety: 224 tensors
  Loaded reasoning: 224 tensors
  ✅ Merged 224 params with {'accuracy': 0.4, 'safety': 0.3, 'reasoning': 0.3}
  ✅ Saved → runs\m6_lora_merging\qwen25_7b_base

✅ M6 default merging complete.


In [4]:
# Cell 3: (Optional) Sweep merge weights
for student_name in STUDENTS:
    specialist_paths = {spec: template.format(student=student_name)
                        for spec, template in SPECIALIST_DIRS.items()}
    if not all(os.path.exists(p) for p in specialist_paths.values()):
        continue

    print(f"\n--- Sweep: {student_name} ---")
    for weights in WEIGHT_SWEEP:
        tag = f"a{weights['accuracy']:.2f}_s{weights['safety']:.2f}_r{weights['reasoning']:.2f}"
        out_path = os.path.join(OUT_DIR, f"{student_name}_sweep_{tag}")
        os.makedirs(out_path, exist_ok=True)

        merged = merge_lora_weights(specialist_paths, weights)
        shutil.copy2(os.path.join(specialist_paths["accuracy"], "adapter_config.json"),
                     os.path.join(out_path, "adapter_config.json"))
        save_file(merged, os.path.join(out_path, "adapter_model.safetensors"))

        tokenizer = AutoTokenizer.from_pretrained(STUDENTS[student_name]["path"], trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
        tokenizer.save_pretrained(out_path)
        print(f"  ✅ {tag}")

print("\n✅ Sweep complete.")


--- Sweep: qwen25_1p5b_base ---
  Loaded accuracy: 224 tensors
  Loaded safety: 224 tensors
  Loaded reasoning: 224 tensors
  ✅ Merged 224 params with {'accuracy': 0.4, 'safety': 0.3, 'reasoning': 0.3}
  ✅ a0.40_s0.30_r0.30
  Loaded accuracy: 224 tensors
  Loaded safety: 224 tensors
  Loaded reasoning: 224 tensors
  ✅ Merged 224 params with {'accuracy': 0.5, 'safety': 0.25, 'reasoning': 0.25}
  ✅ a0.50_s0.25_r0.25
  Loaded accuracy: 224 tensors
  Loaded safety: 224 tensors
  Loaded reasoning: 224 tensors
  ✅ Merged 224 params with {'accuracy': 0.33, 'safety': 0.33, 'reasoning': 0.34}
  ✅ a0.33_s0.33_r0.34
  Loaded accuracy: 224 tensors
  Loaded safety: 224 tensors
  Loaded reasoning: 224 tensors
  ✅ Merged 224 params with {'accuracy': 0.3, 'safety': 0.2, 'reasoning': 0.5}
  ✅ a0.30_s0.20_r0.50

--- Sweep: qwen25_3b_base ---
  Loaded accuracy: 288 tensors
  Loaded safety: 288 tensors
  Loaded reasoning: 288 tensors
  ✅ Merged 288 params with {'accuracy': 0.4, 'safety': 0.3, 'reasoning'